# Ghost systems and free superfermions

`ghost_systems.py` constructs the free field vertex (super)algebras that appear as ghosts, for example in quantum Hamiltonian reduction:

* `BosonicGhosts` (also reached as `vertex_algebras.BosonicGhosts`): even $\beta_i, \gamma_i$ with $\beta_i(z)\gamma_j(w) \sim \delta_{ij}/(z-w)$;
* `FermionicGhosts` (also reached as `vertex_algebras.FermionicGhosts`): odd $b_i, c_i$ with $b_i(z)c_j(w) \sim \delta_{ij}/(z-w)$;
* `FreeSuperFermions`: the fields $\Phi_i$ of a basis of a super vector space with a non-degenerate even form, skew-symmetric on the even part and symmetric on the odd part, $\Phi_i(z)\Phi_j(w) \sim M_{ij}/(z-w)$. These are symplectic bosons and free fermions, Kac–Roan–Wakimoto's neutral free superfermions. The form is given by its Gram matrix (as for lattice vertex algebras), or by a basis and the form as a function.

The λ-brackets and generator names are those of Sage's Lie conformal algebras of the same names (`BosonicGhosts`, `FermionicGhosts`, `Weyl`, `FreeFermions`). Each constructor returns a `UniversalVertexAlgebra`, so equal data give the same algebra.

**Conformal vectors.** None is chosen by default. The common ones are given by name with `conformal_vector=`:

| `conformal_vector=` | weights | βγ: $c$ | bc: $c$ |
|---|---|---|---|
| `'symmetric'` | every field $1/2$ | $-1$ | $1$ |
| `'b1c0'` | $\beta, b$: 1; $\gamma, c$: 0 | $2$ | $-2$ |
| `'b0c1'` | $\beta, b$: 0; $\gamma, c$: 1 | $2$ | $-2$ |
| `'degrees'` + `degrees=` | as given (the two fields of a pair add up to 1) | | |

Also `beta_weight=λ` / `b_weight=λ` (any $\lambda$, or a list with one per pair) gives $c = \pm 2(6\lambda^2 - 6\lambda + 1)$ per pair. Any other conformal vector is an element of the algebra without conformal data. `degrees=` alone gives weights without a conformal vector, and `horizontal_grading='charge'` is the ghost number.

**Prerequisites:** `01_getting_started.ipynb`.

**Contents**

1. The βγ system and its conformal vectors
2. Weight spaces, characters and the charge grading
3. Several pairs
4. The bc system
5. The same algebras as Sage's Lie conformal algebras
6. Free superfermions from a Gram matrix
7. Prescribed weights and general conformal vectors
8. From a basis and a form: the neutral fermions of a reduction
9. Tensor products of ghost systems
10. Unique representation and pickling

In [1]:
from vertex_algebras import *
from vertex_algebras.ghost_systems import BosonicGhosts, FermionicGhosts, FreeSuperFermions
import vertex_algebras as va

## 1. The βγ system and its conformal vectors

Without arguments: the plain algebra, with no conformal data. Its OPEs are $\beta(z)\gamma(w) \sim 1/(z-w)$ and $\gamma(z)\beta(w) \sim -1/(z-w)$.

In [2]:
V = BosonicGhosts(QQ)
print(V)
beta, gamma = V.gens()
print('beta gamma:', beta.ope(gamma), '  gamma beta:', gamma.ope(beta), '  beta beta:', beta.ope(beta))
print('parities:', V.parities(), '  Virasoro vector:', V.conformal_vector())

Universal vertex algebra on generators ['beta', 'gamma'] over Rational Field
beta gamma: [1]   gamma beta: [-1]   beta beta: []
parities: {'beta': 0, 'gamma': 0}   Virasoro vector: None


The three named conformal vectors: their weights and central charges.

In [3]:
for name in ('symmetric', 'b1c0', 'b0c1'):
    W = BosonicGhosts(QQ, conformal_vector=name)
    print('%-10s weights %-28s c = %s   T = %s' % (name, W.degrees(), W.central_charge(), W.conformal_vector()))

symmetric  weights {'beta': 1/2, 'gamma': 1/2}  c = -1   T = -1/2*:dbeta gamma: + 1/2*:dgamma beta:
b1c0       weights {'beta': 1, 'gamma': 0}      c = 2   T = :dgamma beta:
b0c1       weights {'beta': 0, 'gamma': 1}      c = 2   T = -:dbeta gamma:


`beta_weight=λ` for any $\lambda$, with $c = 2(6\lambda^2 - 6\lambda + 1)$. For $\lambda = 3/2$ this is the superghost system of the superstring, $c = 11$. A symbolic $\lambda$ works over $\mathbb Q(\lambda)$.

In [4]:
print([(l, BosonicGhosts(QQ, beta_weight=l).central_charge()) for l in (0, 1/2, 1, 3/2, 2, -1/3)])
K.<l> = Frac(QQ['l'])
Vl = BosonicGhosts(K, beta_weight=l)
print(Vl.central_charge(), Vl.degrees())

[(0, 2), (1/2, -1), (1, 2), (3/2, 11), (2, 26), (-1/3, 22/3)]
12*l^2 - 12*l + 2 {'beta': l, 'gamma': -l + 1}


## 2. Weight spaces, characters and the charge grading

With weights $1/2$ every weight space is finite-dimensional, and the character is $\prod_{n\ge1} (1-q^{n-1/2})^{-2}$.

In [5]:
V12 = BosonicGhosts(QQ, conformal_vector='symmetric')
print(V12.character(4))
P.<t> = PowerSeriesRing(QQ, default_prec=8)          # t = q^(1/2)
print([V12.weight_space(k / 2).dimension() for k in range(8)], list(prod((1 - t^(2*n - 1))^-2 for n in range(1, 5)))[:8])

1 + 2*q^(1/2) + 3*q + 6*q^(3/2) + 9*q^2 + 14*q^(5/2) + 22*q^3 + 32*q^(7/2) + O(q^4)
[1, 2, 3, 6, 9, 14, 22, 32] [1, 2, 3, 6, 9, 14, 22, 32]


With `'b1c0'`, $\gamma$ has weight 0, so the weight spaces are infinite-dimensional (the powers of $\gamma$). The ghost charge grading (`horizontal_grading='charge'`: $\gamma, c$ have charge $+1$, $\beta, b$ charge $-1$) makes them finite for each charge.

In [6]:
V1 = BosonicGhosts(QQ, conformal_vector='b1c0')
print(V1.weight_space(1).is_finite_dimensional())
V1c = BosonicGhosts(QQ, conformal_vector='b1c0', horizontal_grading='charge')
print(V1c.gen('gamma').horizontal_grade(), V1c.gen('beta').horizontal_grade())
print([V1c.weight_space(1, horizontal_grade=g).dimension() for g in (-2, -1, 0, 1, 2)])
print(V1c.weight_space(1, horizontal_grade=0).basis())

False
1 -1
[0, 1, 1, 2, 2]
[:beta gamma:]


At charge 0 a pair contributes weight $(\lambda + n) + (1 - \lambda + m) = 1 + n + m$, so the dimensions of the charge-0 weight spaces do not depend on $\lambda$.

In [7]:
for name in ('symmetric', 'b1c0', 'b0c1'):
    W = BosonicGhosts(QQ, conformal_vector=name, horizontal_grading='charge')
    print('%-10s' % name, [W.weight_space(n, horizontal_grade=0).dimension() for n in range(7)])

symmetric  [1, 1, 3, 6, 12, 21, 38]
b1c0       [1, 1, 3, 6, 12, 21, 38]
b0c1       [1, 1, 3, 6, 12, 21, 38]


## 3. Several pairs

`rank=n` gives $n$ pairs. The default names are Sage's, `beta0, ..., gamma0, ...`; `names=` lists the first fields of all pairs, then the second ones. The weights can differ from pair to pair: give a list to `beta_weight=`, or a dict with `conformal_vector='degrees'`.

In [8]:
V3 = BosonicGhosts(QQ, rank=3, beta_weight=[0, 1/2, 1])
print(V3.variable_names(), V3.central_charge())
print(V3.degrees())
Vw = BosonicGhosts(QQ, rank=3, conformal_vector='degrees',
                   degrees={'beta0': 0, 'beta1': 1/2, 'beta2': 1, 'gamma0': 1, 'gamma1': 1/2, 'gamma2': 0})
print(Vw is V3)
Vn = BosonicGhosts(QQ, rank=2, names=['u', 'v', 'x', 'y'], conformal_vector='symmetric')
print(Vn.gen('u').ope(Vn.gen('x')), Vn.gen('u').ope(Vn.gen('y')))

('beta0', 'beta1', 'beta2', 'gamma0', 'gamma1', 'gamma2') 3
{'beta0': 0, 'beta1': 1/2, 'beta2': 1, 'gamma0': 1, 'gamma1': 1/2, 'gamma2': 0}
True
[1] []


## 4. The bc system

The fermionic version: $b(z)c(w) \sim 1/(z-w) \sim c(z)b(w)$, with $c = -2(6\lambda^2 - 6\lambda + 1)$. For $\lambda = 2$ these are the ghosts of the bosonic string, $c = -26$. At charge 0 the character is $1/(q)_\infty$ for any $\lambda$: the boson–fermion correspondence.

In [9]:
B = FermionicGhosts(QQ)
b, c = B.gens()
print('b c:', b.ope(c), '  c b:', c.ope(b), '  b b:', b.ope(b), '  parities:', B.parities())
print([(name, FermionicGhosts(QQ, conformal_vector=name).central_charge()) for name in ('symmetric', 'b1c0', 'b0c1')])
print([(l, FermionicGhosts(QQ, b_weight=l).central_charge()) for l in (3/2, 2, 3)])
B2 = FermionicGhosts(QQ, b_weight=2, horizontal_grading='charge')
print(B2.degrees(), B2.character(7, horizontal_grade=0))
print(va.FermionicGhosts is FermionicGhosts, va.BosonicGhosts is BosonicGhosts)

b c: [1]   c b: [1]   b b: []   parities: {'b': 1, 'c': 1}
[('symmetric', 1), ('b1c0', -2), ('b0c1', -2)]
[(3/2, -11), (2, -26), (3, -74)]
{'b': 2, 'c': -1} 1 + q + 2*q^2 + 3*q^3 + 5*q^4 + 7*q^5 + 11*q^6 + O(q^7)
True True


The Zhu algebra of the $bc$ system with weights 1 and 0 is the Clifford algebra on $[b], [c]$.

In [10]:
A = FermionicGhosts(QQ, conformal_vector='b1c0').zhu_algebra()
zb, zc = A.gens()
zb * zc + zc * zb, zb * zb

(1, 0)

## 5. The same algebras as Sage's Lie conformal algebras

The universal enveloping vertex algebra of each of Sage's examples, with $K = 1$, is the *same object* as ours. Sage's degrees give $\beta$ and $b$ weight 1.

In [11]:
from vertex_algebras.lie_conformal import universal_enveloping_vertex_algebra as U
K1 = {'K': 1}
print(U(lie_conformal_algebras.BosonicGhosts(QQ), central_parameters=K1)
      is BosonicGhosts(QQ, degrees={'beta': 1, 'gamma': 0}))
print(U(lie_conformal_algebras.FermionicGhosts(QQ), central_parameters=K1)
      is FermionicGhosts(QQ, degrees={'b': 1, 'c': 0}))
print(U(lie_conformal_algebras.Weyl(QQ), central_parameters=K1)
      is FreeSuperFermions(QQ, matrix([[0, 1], [-1, 0]]), names=['alpha0', 'alpha1']))
M = matrix([[0, 1], [1, 0]])
print(U(lie_conformal_algebras.FreeFermions(QQ, gram_matrix=M), central_parameters=K1)
      is FreeSuperFermions(QQ, M, names=['psi_0', 'psi_1'], degrees={'psi_0': 1/2, 'psi_1': 1/2}))

True
True
True
True


## 6. Free superfermions from a Gram matrix

As for lattice vertex algebras, the parities are read off the Gram matrix. A block of basis vectors connected by nonzero entries is even if $M$ is skew-symmetric on it (symplectic bosons) and odd if symmetric (fermions). The default names are `Phi0, Phi1, ...`; `names=` takes a prefix or a list. With `'symmetric'`, every field has weight $1/2$ and $c = -\operatorname{sdim}/2$.

In [12]:
examples = {'Majorana fermion': matrix([[1]]), 'symplectic pair': matrix([[0, 1], [-1, 0]]),
            'fermion pair': matrix([[0, 1], [1, 0]]),
            'mixed, any order': matrix([[0, 0, 1, 0], [0, 2, 0, 0], [-1, 0, 0, 0], [0, 0, 0, -3]])}
for name, M in examples.items():
    F = FreeSuperFermions(QQ, M, conformal_vector='symmetric')
    print('%-18s parities %-44s c = %s' % (name, F.parities(), F.central_charge()))

Majorana fermion   parities {'Phi0': 1}                                  c = 1/2
symplectic pair    parities {'Phi0': 0, 'Phi1': 0}                       c = -1
fermion pair       parities {'Phi0': 1, 'Phi1': 1}                       c = 1
mixed, any order   parities {'Phi0': 0, 'Phi1': 1, 'Phi2': 0, 'Phi3': 1} c = 0


The symmetric conformal vector of a symplectic pair, and of a fermion pair, is exactly the βγ system, and the bc system, with weights $1/2$.

In [13]:
J, S = matrix([[0, 1], [-1, 0]]), matrix([[0, 1], [1, 0]])
print(FreeSuperFermions(QQ, J, names=['beta', 'gamma'], conformal_vector='symmetric')
      is BosonicGhosts(QQ, conformal_vector='symmetric'))
print(FreeSuperFermions(QQ, S, names=['b', 'c'], conformal_vector='symmetric')
      is FermionicGhosts(QQ, conformal_vector='symmetric'))
F = FreeSuperFermions(QQ, matrix([[0, 2], [-2, 0]]), names='x', conformal_vector='symmetric')
print(F.variable_names(), F.gen('x0').ope(F.gen('x1')), F.conformal_vector())

True
True
('x0', 'x1') [2] -1/4*:dx0 x1: + 1/4*:dx1 x0:


Forms that are not admissible are refused: one that is neither skew-symmetric nor symmetric on a block, one mixing both on a connected block (here $\Phi_0, \Phi_1$ skew and $\Phi_0, \Phi_2$ symmetric), and a degenerate one.

In [14]:
for M in (matrix([[0, 1], [2, 0]]), matrix([[0, 1, 1], [-1, 0, 0], [1, 0, 1]]), matrix([[1, 1], [1, 1]])):
    try:
        FreeSuperFermions(QQ, M)
    except ValueError as err:
        print('ValueError:', err)

ValueError: the form must be, on each block of basis vectors it connects, skew-symmetric (even) or symmetric (odd)
ValueError: the form must be, on each block of basis vectors it connects, skew-symmetric (even) or symmetric (odd)
ValueError: the form must be non-degenerate


## 7. Prescribed weights and general conformal vectors

`conformal_vector='degrees'` with `degrees=` builds the conformal vector with exactly those weights, $T = T_{\mathrm{sym}} + \partial J$ with $J$ quadratic. The weights of paired fields ($M_{ij} \ne 0$) must add up to 1, so a Majorana fermion has weight $1/2$. Here a βγ pair with $\lambda = 2$, a bc pair with $\lambda = 3/2$ and a Majorana fermion: $c = 26 - 11 + 1/2$.

In [15]:
Mb = block_diagonal_matrix([J, S, matrix([[1]])])
w = {'Phi0': 2, 'Phi1': -1, 'Phi2': 3/2, 'Phi3': -1/2, 'Phi4': 1/2}
Wf = FreeSuperFermions(QQ, Mb, conformal_vector='degrees', degrees=w)
print(Wf.degrees(), Wf.central_charge())
try:
    FreeSuperFermions(QQ, Mb, conformal_vector='degrees', degrees=dict(w, Phi4=1))
except ValueError as err:
    print('ValueError:', err)

{'Phi0': 2, 'Phi1': -1, 'Phi2': 3/2, 'Phi3': -1/2, 'Phi4': 1/2} 31/2
ValueError: the weights of paired fields (M_ij ≠ 0) must add up to 1


Any other conformal vector, as for lattice vertex algebras, is an element of the algebra without conformal data. A hand-built conformal vector on a symplectic pair gives the βγ system with $\beta$ of weight $3/2$.

In [16]:
P0 = FreeSuperFermions(QQ, J, names=['beta', 'gamma'])
pb, pg = P0.gens()
T = pb.derivative().no(pg) / 2 + 3/2 * pb.no(pg.derivative())
G = FreeSuperFermions(QQ, J, names=['beta', 'gamma'], conformal_vector=T)
print(G.degrees(), G.central_charge(), G is BosonicGhosts(QQ, beta_weight=3/2))

{'beta': 3/2, 'gamma': -1/2} 11 True


## 8. From a basis and a form: the neutral fermions of a reduction

`FreeSuperFermions(R, basis, form)` takes elements of a super module and the form as a function. The parities are those of the elements, checked against the form, and the names are the elements as they print.

In the quantum Hamiltonian reduction of Kac–Roan–Wakimoto these are the neutral free superfermions on $\mathfrak g_{1/2}$, with $\langle a, b\rangle = (f\,|\,[a, b])$. Take the minimal grading $x = \theta/2$, so $\mathfrak g_{1/2}$ is spanned by the root vectors of the roots $\beta$ with $(\beta|\theta) = 1$, and $f = e_{-\theta}$. For $\mathfrak{sl}(2|1)$ this gives two fermions (the $N = 2$ superconformal algebra), and for $\mathfrak{sl}_3$ two symplectic bosons (Bershadsky–Polyakov).

In [17]:
from vertex_algebras.lie_superalgebras import LieSuperalgebra


def neutral_fermions(g):
    th = g.theta()
    f = g.root_vector(th, -1)
    half = [g.basis()[k] for k in g.basis().keys()
            if any(g.root_on_basis(k)) and g.root_norm(g.root_on_basis(k), th) == 1]
    return half, FreeSuperFermions(QQ, half, lambda a, b: g.invariant_form(f, a.bracket(b)),
                                   conformal_vector='symmetric')


g = LieSuperalgebra(QQ, 'sl(2|1)')
half, Ne = neutral_fermions(g)
print(half)
print(Ne.parities(), Ne.central_charge())
print([x.ope(y) for x in Ne.gens() for y in Ne.gens()])

[E[alpha[1] + alpha[2]], E[-alpha[2]]]
{'E[alpha[1] + alpha[2]]': 1, 'E[-alpha[2]]': 1} 1
[[], [-1], [-1], []]


Further minimal reductions. The dimensions of $\mathfrak g_{1/2}$ are those of Kac–Wakimoto's tables. The form $(f|[\cdot,\cdot])$ is non-degenerate, and $c = -\operatorname{sdim}\mathfrak g_{1/2}/2$ is KRW's contribution of the neutral fermions to the central charge.

In [18]:
for label, spec, kw in [('sl_3 (Bershadsky-Polyakov)', CartanMatrix(['A', 2]), {}),
                        ('osp(1|2) = spo(2|1)', 'spo(2|1)', {}), ('D(2,1;2)', 'D(2,1;a)', {'a': 2}),
                        ('F(4), θ in sl_2', 'F(4)', {'theta': 'sl2'}), ('G(3), θ in sl_2', 'G(3)', {'theta': 'sl2'}),
                        ('osp(5|2)', 'osp(5|2)', {})]:
    g = LieSuperalgebra(QQ, spec, **kw)
    half, Ne = neutral_fermions(g)
    even = sum(1 for x in half if not x.is_even_odd())
    print('%-28s g_1/2: %d even, %d odd   c = %s' % (label, even, len(half) - even, Ne.central_charge()))

sl_3 (Bershadsky-Polyakov)   g_1/2: 2 even, 0 odd   c = -1
osp(1|2) = spo(2|1)          g_1/2: 0 even, 1 odd   c = 1/2


D(2,1;2)                     g_1/2: 0 even, 4 odd   c = 2


F(4), θ in sl_2              g_1/2: 0 even, 8 odd   c = 4


G(3), θ in sl_2              g_1/2: 0 even, 7 odd   c = 7/2
osp(5|2)                     g_1/2: 2 even, 4 odd   c = 1


## 9. Tensor products of ghost systems

Ghost systems combine with other vertex algebras. The superstring's ghosts: the βγ system with $\lambda = 3/2$ ($c = 11$) and the bc system with $\lambda = 2$ ($c = -26$), together $c = -15$.

In [19]:
ghosts = BosonicGhosts(QQ, beta_weight=3/2).tensor(FermionicGhosts(QQ, b_weight=2), conformal_vector='factors')
print(ghosts.central_charge(), ghosts.degrees())

-15 {'beta': 3/2, 'gamma': -1/2, 'b': 2, 'c': -1}


## 10. Unique representation and pickling

In [20]:
print(BosonicGhosts(QQ, conformal_vector='b1c0') is BosonicGhosts(QQ, beta_weight=1))
print(loads(dumps(Wf)) is Wf, loads(dumps(Ne)) is Ne)
TestSuite(Wf).run(); TestSuite(B2).run()
print('TestSuite ok')

True
True True


TestSuite ok


## References

- [FMS1986] D. Friedan, E. Martinec, S. Shenker, *Conformal invariance, supersymmetry and string theory*, Nuclear Phys. B 271 (1986), 93–165, [doi:10.1016/0550-3213(86)90356-1](https://doi.org/10.1016/0550-3213(86)90356-1). — βγ and bc systems of weight λ, the ghosts of the superstring.
- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015). — neutral free superfermions in the reduction, their central charge (section 8).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011). — the tables of minimal reductions (section 8).
- [Ber1991] M. Bershadsky, *Conformal field theories via Hamiltonian reduction*, Comm. Math. Phys. 139 (1991), 71–82, [doi:10.1007/BF02102729](https://doi.org/10.1007/BF02102729).
- [Pol1990] A. M. Polyakov, *Gauge transformations and diffeomorphisms*, Internat. J. Modern Phys. A 5 (1990), 833–842, [doi:10.1142/S0217751X90000386](https://doi.org/10.1142/S0217751X90000386). — the Bershadsky–Polyakov algebra.